# Diagnostics forward models: results

Every result of the diagnostics-forward-models area, saved into this directory.

1. **Thomson-scattering validation** at the pinned design point: the spectral density, the layout, a channel's signal against its lens standoff, and its relative uncertainty.
2. **Spectral band optimization**: the bands scaled by one factor `s` per volume (`band_edges`), chosen by the Fisher-information rule.
3. **Interferometer chords**: the pinned 5 parallel chords.
4. **Cross-sections of the thesis**: the two layouts above, drawn side by side at print size.

In [ ]:
import functools
import pathlib

import jax.numpy as jnp
import matplotlib.lines
import matplotlib.transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.bayesian_inference import (  # noqa: E501
    desc_truth_types,
    profile_parametrization_types,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    diagnostic_design_point,
    geometry_utils,
    interferometer_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    thomson_scattering_types as ts_types,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    thomson_scattering_utils as ts,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    pubfig,
    synthetic_target_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_diagnostics_forward_models as plots,
)

# Anchor the output directory to the package location, not the kernel's cwd, so the
# notebook runs from any working directory.
OUTPUT_DIR = (
    pathlib.Path(plots.__file__).parents[1] / "results" / "diagnostics_forward_models"
)
assert OUTPUT_DIR.is_dir(), OUTPUT_DIR
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)


def save_pdf(figure, name: str) -> None:
    figure.savefig(PLOTS_DIR / f"{name}.pdf")


pubfig.apply_style()
design = diagnostic_design_point.DiagnosticDesignPoint()
DATA_DIRECTORY = pathlib.Path(plots.__file__).parents[4] / "data"
DATASET = dataset_utils.DEFAULT_DATASET
truth = desc_truth_types.read_desc_truth(
    "wide_high_beta_gp",
    DATA_DIRECTORY / f"{DATASET}.parquet",
    DATA_DIRECTORY / f"{DATASET}_settings.json",
)
reference = desc_truth_types.read_desc_truth(
    desc_truth_types.REFERENCE_EQUILIBRIUM_ID,
    DATA_DIRECTORY / f"{DATASET}.parquet",
    DATA_DIRECTORY / f"{DATASET}_settings.json",
)
placement = synthetic_target_utils.build_diagnostic_placement(design, reference)
geometry = placement.thomson_geometry
response = placement.polychromator_response
chords = placement.interferometer_chords
electron_density_fn = functools.partial(
    profile_parametrization_types.electron_density, truth.profiles
)
electron_temperature_fn = functools.partial(
    profile_parametrization_types.electron_temperature, truth.profiles
)
band_labels = [
    f"{(center - width) * 1e9:.0f}-{(center + width) * 1e9:.0f} nm"
    for center, width in zip(response.filter_centers, response.filter_widths)
]
print(
    f"volumes: {np.asarray(geometry.volume_positions).shape[0]}, "
    f"optics: {np.asarray(geometry.optic_positions).shape[0]}, "
    f"interferometer chords: {len(chords)}"
)
print(f"channels: {band_labels}")
print(
    f"noise model: {design.thomson_noise_model}, kappa = "
    f"{placement.thomson_signal_scale:.3e} V s per photoelectron"
)

## 1a. The scattering function

Area-normalized relativistic spectral density (Naito et al. 1993, eq. 3) at `theta = 150 deg`, for `Te` from 100 eV to 10 keV. As `Te` rises the spectrum widens and its peak moves blue-ward of the 1064 nm laser line, which is what `s` tracks. The cell prints the peak positions at both ends of the range.

In [ ]:
laser_wavelength = float(response.laser_wavelength)
wavelength_nm = jnp.linspace(300.0, 1600.0, 2001)
wavelength = wavelength_nm * 1e-9
scattering_angle_degrees = 150.0
spectrum_temperatures_ev = (100.0, 1.0e3, 5.0e3, 1.0e4)
peak_temperatures_ev = jnp.linspace(
    min(spectrum_temperatures_ev), max(spectrum_temperatures_ev), 40
)
# Area-normalized, so the shapes compare across temperatures.
normalized_spectra = []
for temperature_ev in [*spectrum_temperatures_ev, *peak_temperatures_ev]:
    spectrum = ts.spectral_density_relativistic(
        wavelength,
        jnp.radians(scattering_angle_degrees),
        temperature_ev,
        laser_wavelength,
    )
    normalized_spectra.append(
        spectrum * laser_wavelength / jnp.trapezoid(spectrum, wavelength)
    )

figure, axes = pubfig.subplots(1, 1, width="single", aspect=0.8)
axis = axes.flat[0]
for temperature_ev, spectrum, color in zip(
    spectrum_temperatures_ev, normalized_spectra, pubfig.PALETTE
):
    axis.plot(
        wavelength_nm,
        spectrum,
        lw=1.0,
        color=color,
        label=f"{temperature_ev / 1e3:g} keV",
    )
axis.axvline(laser_wavelength * 1e9, color="0.5", lw=0.7, ls=":")
axis.set_xlabel(r"$\lambda$ (nm)")
axis.set_ylabel(r"$S$ (normalised)")
axis.legend()
save_pdf(figure, "ts_validation_spectral_density")
peak_wavelengths_nm = [
    float(wavelength_nm[int(jnp.argmax(spectrum))])
    for spectrum in normalized_spectra[len(spectrum_temperatures_ev) :]
]
spectral_summary = {
    "peak_cold_nm": peak_wavelengths_nm[0],
    "peak_hot_nm": peak_wavelengths_nm[-1],
}
print(
    f"peak position at theta = 150 deg: {spectral_summary['peak_cold_nm']:.0f} nm at "
    f"100 eV -> {spectral_summary['peak_hot_nm']:.0f} nm at 10 keV "
    f"(laser at {response.laser_wavelength * 1e9:.0f} nm)"
)
assert spectral_summary["peak_cold_nm"] < response.laser_wavelength * 1e9
assert spectral_summary["peak_hot_nm"] < spectral_summary["peak_cold_nm"], (
    "the spectrum peak must blue-shift with temperature"
)

## 1b. The design-point layout

The pinned layout at `zeta = 0` on the `wide_high_beta_gp` DESC truth: 15 volumes along the horizontal laser, read by a forward-scatter optic (viewing the high-field side) and a back-scatter optic (the low-field side), and, in a second figure, the collected light per volume and its uncertainty. An optic outside its scattering-angle range would be ringed in red. The optic surface angles (`-150 deg`, `+20 deg`) are pinned to this laser and do not carry over to a tilted one.

In [ ]:
signals, uncertainties = synthetic_target_utils.readings_on_layout(
    truth.coefficients,
    geometry,
    response,
    design,
    placement.thomson_signal_scale,
    electron_density_fn,
    electron_temperature_fn,
)
# The cross-section and the collected light as separate figures, so the cross-section
# can stand alone. Plain figures, since the helper lays out its own colorbar.
layout_figure, layout_axes = plt.subplots(figsize=(6.5, 6.0))
light_figure, light_axes = plt.subplots(figsize=(7.0, 6.0))
plots.plot_chord_layout(
    truth.coefficients,
    geometry,
    signals,
    uncertainties,
    surface_offset=design.surface_offset,
    axes_pair=(layout_axes, light_axes),
)
save_pdf(layout_figure, "ts_validation_chord_layout")
save_pdf(light_figure, "ts_validation_collected_light")

## 1c. Signal and uncertainty against the lens standoff

Two volumes at `rho ~ 0.7`, viewed at 15 and 30 deg (forward optic) and at 150 and 165 deg (back-scatter optic), each lens retracted along its line of sight from 0.3 to 3 m. The bands are the published W7-X ones, on which `kappa` is calibrated. Drawn is the second-largest channel, the weakest reading a spectral fit needs. A cross marks where the SNR drops through 2.

`Te` is the `wide_high_beta_gp` truth's at each volume. The density is lowered to `2e16 m^-3`, far below the `5e18 m^-3` the system is calibrated to detect, so every curve starts on the 10% floor of the noise curve and the faintest fall through SNR 2 within the scan: a probe of the detector model, not a plasma scenario. At the truth's own density the error bar is a flat 10% at any standoff. The second figure shows the truth's `ne` and `Te` profiles with the two volumes marked.

In [ ]:
rho = synthetic_target_utils.thomson_volume_flux_labels(
    geometry.volume_positions, reference
)
# Expected density at every volume, for the band cases of section 2.
expected_density = synthetic_target_utils.expected_volume_state(geometry, reference)[0]
# The truth's Te at each volume, each mapped to rho on the truth's own geometry as
# the inference does, so the probe reads the Te the inference studies see.
truth_rho = synthetic_target_utils.thomson_volume_flux_labels(
    geometry.volume_positions, truth
)
truth_temperature = np.asarray(electron_temperature_fn(jnp.asarray(truth_rho)))
served = np.asarray(geometry.volume_optic_index)
# Index 0 is the forward-scatter set, 1 the back-scatter set; one volume from each.
forward = int(np.flatnonzero(served == 0)[np.argmin(np.abs(rho[served == 0] - 0.7))])
backward = int(np.flatnonzero(served == 1)[np.argmin(np.abs(rho[served == 1] - 0.7))])
curves = ((forward, 15.0), (forward, 30.0), (backward, 150.0), (backward, 165.0))
THIN_DENSITY_M3 = 2.0e16
"""Probe density: curves start at the 10% floor and the faintest cross SNR 2."""
mantissa, exponent = f"{THIN_DENSITY_M3:.1e}".split("e")
density_title = (
    rf"$n_e = {float(mantissa):g} \times 10^{{{int(exponent)}}}$" + r" m$^{-3}$"
)
# The two volumes differ in Te, so the title names both.
temperature_title = "$T_e = " + ", $".join(
    rf"{truth_temperature[volume] / 1e3:.1f}$ keV ($\theta = "
    + ", ".join(f"{angle:.0f}" for index, angle in curves if index == volume)
    + r"^\circ$)"
    for volume in (forward, backward)
)
state_title = f"{density_title}, {temperature_title}"
# The published bands, on which kappa is calibrated.
w7x_response = response.model_copy(update={"band_scales": None})
distances_m = jnp.linspace(0.3, 3.0, 30)
volume_positions = jnp.asarray(geometry.volume_positions)
design_optics = jnp.asarray(geometry.optic_positions)[
    jnp.asarray(geometry.volume_optic_index)
]
laser = jnp.asarray(geometry.laser_direction)
laser = laser / jnp.linalg.norm(laser)
# The in-plane normal to the beam, so a line of sight at angle theta from the beam
# stays in the poloidal plane the layout lives in.
normal = jnp.array([0.0, 0.0, 1.0])
normal = normal - jnp.dot(normal, laser) * laser
normal = normal / jnp.linalg.norm(normal)

figure, axes = pubfig.subplots(1, 1, width="single", aspect=0.85)
axis = axes.flat[0]
standoff_rows = []
for (volume_index, angle_degrees), color in zip(curves, pubfig.PALETTE):
    angle = jnp.radians(angle_degrees)
    view = jnp.cos(angle) * laser + jnp.sin(angle) * normal
    volume = volume_positions[volume_index]
    # The layout reduced to this one volume, its lens at each distance along the view.
    signals_per_distance = []
    for distance in distances_m:
        single_volume_geometry = ts_types.ThomsonScatteringGeometry(
            laser_origin=jnp.asarray(geometry.laser_origin),
            laser_direction=jnp.asarray(geometry.laser_direction),
            reference_phi=geometry.reference_phi,
            volume_positions=volume[None, :],
            optic_positions=(volume + distance * view)[None, :],
            volume_optic_index=jnp.zeros(1, dtype=jnp.int64),
            collection_aperture_diameter=geometry.collection_aperture_diameter,
            collection_focal_length=geometry.collection_focal_length,
            fiber_core_diameter=geometry.fiber_core_diameter,
        )
        signals_per_distance.append(
            ts.thomson_signals(
                single_volume_geometry,
                w7x_response,
                jnp.asarray([THIN_DENSITY_M3]),
                jnp.asarray([float(truth_temperature[volume_index])]),
                ts.absolute_calibration(single_volume_geometry),
            )[0]
        )
    line_signals = jnp.stack(signals_per_distance)
    line_sigma = ts.signal_uncertainties(
        line_signals, design.thomson_noise_model, placement.thomson_signal_scale
    )
    # The second largest reading, the weakest one a spectral fit needs.
    channel = int(jnp.argsort(line_signals[0])[-2])
    reading, error = line_signals[:, channel], line_sigma[:, channel]
    snr = jnp.where(error > 0.0, reading / jnp.where(error > 0.0, error, 1.0), 0.0)
    below = jnp.flatnonzero(snr < ts.DETECTION_SNR)
    crossing = float(distances_m[below[0]]) if below.size else float("nan")
    # A channel expecting under one photoelectron per pulse collects no light
    # and has nothing to draw on a log axis.
    collects_light = np.asarray(reading >= 1.0)
    axis.errorbar(
        np.asarray(distances_m)[collects_light],
        np.asarray(reading)[collects_light],
        yerr=np.asarray(error)[collects_light],
        fmt="-o",
        ms=2.5,
        lw=1.0,
        capsize=1.5,
        color=color,
        label=rf"$\theta = {angle_degrees:.0f}^\circ$",
    )
    if below.size and collects_light[int(below[0])]:
        axis.plot(crossing, float(reading[below[0]]), "x", ms=8, mew=1.5, color=color)
    standoff_rows.append(
        {
            "state": state_title,
            "volume": volume_index,
            "theta_deg": angle_degrees,
            "channel": channel + 1,
            "design_standoff_m": float(
                jnp.linalg.norm(design_optics[volume_index] - volume)
            ),
            "snr_below_threshold_from_m": crossing,
        }
    )
axis.set_title(f"{density_title}\n{temperature_title}", fontsize="small")
axis.set_xlabel("lens-to-volume distance $d$ (m)")
axis.set_yscale("log")
axis.set_ylabel(r"$D_{ij}$ (photoelectrons)")
# The cross is described in the caption, so the legend carries the curves alone
# and fits inside the axes.
axis.legend(fontsize="small", loc="upper right")
save_pdf(figure, "ts_validation_signal_vs_standoff")

# The truth the probe takes its Te from, with the two probed volumes marked.
rho_grid = jnp.linspace(0.0, 1.0, 200)
truth_panels = (
    (
        np.asarray(electron_density_fn(rho_grid)),
        np.asarray(electron_density_fn(jnp.asarray(truth_rho))),
        r"$n_e$ [m$^{-3}$]",
    ),
    (
        np.asarray(electron_temperature_fn(rho_grid)) / 1e3,
        truth_temperature / 1e3,
        r"$T_e$ [keV]",
    ),
)
figure, axes = pubfig.subplots(1, 2, width="double", aspect=0.8)
for axis, (profile, at_volumes, axis_label) in zip(axes.flat, truth_panels):
    axis.plot(
        np.asarray(rho_grid),
        profile,
        color="k",
        lw=1.2,
    )
    for volume, marker in ((forward, "o"), (backward, "s")):
        angles = ", ".join(f"{angle:.0f}" for index, angle in curves if index == volume)
        axis.plot(
            truth_rho[volume],
            at_volumes[volume],
            marker,
            color="k",
            ms=5,
            label=rf"volume at $\theta = {angles}^\circ$",
        )
    axis.set_ylim(bottom=0.0)
    axis.set_xlabel(r"$\rho$")
    axis.set_ylabel(axis_label)
axes.flat[0].legend(fontsize="small", loc="lower left")
save_pdf(figure, "ts_validation_standoff_truth_profiles")
standoff_table = pd.DataFrame(standoff_rows)
for v in (forward, backward):
    print(
        f"volume {v}: rho {truth_rho[v]:.3f}, truth Te {truth_temperature[v]:.0f} eV, "
        f"probe ne {THIN_DENSITY_M3:.1e} m^-3"
    )
print(standoff_table.to_string(index=False))

## 1d. Relative uncertainty `sigma_i / D_i` against the channel signal `D_i`

The two noise models in the units of the forward model: the W7-X curve of Raak (2024) Fig. 6, mapped through `I = kappa D`. The counting model falls as `1/sqrt(D_i)`. The electronic model saturates at `c = 10%` for large signals, this work's value above the about 8.7% at which Fig. 6 levels off, and reaches `sigma_i = D_i` at the floor `sigma_floor = I_floor / kappa`. The dashed line is the `SNR = 2` threshold.

The curves are fixed by the detector calibration, so a placement that collects less light moves its readings to the left along them. The cell prints the range the pinned layout reads over.

In [ ]:
noise_floor = ts.NOISE_FLOOR_INTEGRATED_SIGNAL / placement.thomson_signal_scale
lit = np.asarray(signals)[np.asarray(signals) > 0.0]
# From a decade below the floor to a decade above the layout's brightest reading.
signal_grid = np.asarray(
    jnp.logspace(np.log10(0.1 * noise_floor), np.log10(10.0 * float(lit.max())), 400)
)
relative_uncertainty_figure, axes = pubfig.subplots(1, 1, width="single", aspect=0.9)
axis = axes.flat[0]
axis.set_xscale("log")
# The counting model masks its faint channels, so its curve is evaluated directly
# rather than through the mask, which would leave a hole at the low-signal end.
axis.plot(
    signal_grid,
    1.0 / np.sqrt(signal_grid),
    color=pubfig.CATEGORICAL_PALETTE[0],
    lw=1.3,
    label=r"counting, $1/\sqrt{D_{ij}}$",
)
axis.plot(
    signal_grid,
    np.asarray(
        ts.signal_uncertainties(
            signal_grid, "electronic", placement.thomson_signal_scale
        )
    )
    / signal_grid,
    color=pubfig.CATEGORICAL_PALETTE[1],
    lw=1.3,
    label="electronic",
)
axis.axhline(1.0 / ts.DETECTION_SNR, color="0.4", ls="--", lw=0.8)
axis.axhline(1.0, color="0.4", ls=":", lw=0.8)
axis.axvline(noise_floor, color="0.55", ls="-.", lw=0.8)
handles, legend_labels = axis.get_legend_handles_labels()
handles += [
    matplotlib.lines.Line2D([], [], color="0.4", ls="--", lw=0.8),
    matplotlib.lines.Line2D([], [], color="0.4", ls=":", lw=0.8),
    matplotlib.lines.Line2D([], [], color="0.55", ls="-.", lw=0.8),
]
legend_labels += [
    r"detectable, $\mathrm{SNR} = 2$",
    r"$\sigma_{ij} = D_{ij}$",
    r"$D_{ij} = \sigma_{\mathrm{floor}}$",
]
axis.set_xlim(float(signal_grid.min()), float(signal_grid.max()))
# The electronic curves diverge at the low-signal end, so the linear axis is cut
# just above `sigma = D`, where they leave the frame at the floor.
axis.set_ylim(0.0, 1.1)
axis.set_xlabel(r"channel signal $D_{ij}$ (photoelectrons)")
axis.set_ylabel(r"relative uncertainty $\sigma_{ij} / D_{ij}$")
axis.legend(
    handles=handles, labels=legend_labels, loc="upper right", fontsize="x-small"
)
save_pdf(relative_uncertainty_figure, "ts_validation_relative_uncertainty")
print(
    f"noise floor {noise_floor:.3g} photoelectrons per channel, from I_floor = "
    f"{ts.NOISE_FLOOR_INTEGRATED_SIGNAL:.3g} V s"
)
print(
    f"layout readings {lit.min():.3g} to {lit.max():.3g} photoelectrons, "
    "relative uncertainty "
    f"{float(np.median(uncertainties[signals > 0] / lit)):.3f} median"
)

## 2. Spectral band optimization: the Fisher-information rule

The rule (`fisher_band_scale`, the design point's default) picks the `s` in `band_scale_bounds` that minimizes the Cramer-Rao variance of `ln Te` at a volume's expected state. Six cases: the two volumes of section 1c at 100 eV, 1 keV and 10 keV, each at its expected density. Each panel shows the normalized spectrum, the five channels at the selected `s`, and `s` itself. An `s` on a bound of `[0.50, 1.16]` is set by the bound, not by the state. The CSV records the band edges of every case.

In [ ]:
CASE_TEMPERATURES_EV = (100.0, 1.0e3, 1.0e4)
case_volumes = [forward] * len(CASE_TEMPERATURES_EV) + [backward] * len(
    CASE_TEMPERATURES_EV
)
case_temperatures = np.array(list(CASE_TEMPERATURES_EV) * 2)
case_density = expected_density[np.array(case_volumes)]
positions = np.asarray(geometry.volume_positions)
# The two volumes repeated once per temperature, each with its own design optic.
case_geometry = ts_types.ThomsonScatteringGeometry(
    laser_origin=geometry.laser_origin,
    laser_direction=geometry.laser_direction,
    reference_phi=geometry.reference_phi,
    volume_positions=positions[case_volumes],
    optic_positions=geometry.optic_positions,
    volume_optic_index=served[case_volumes],
    collection_aperture_diameter=geometry.collection_aperture_diameter,
    collection_focal_length=geometry.collection_focal_length,
    fiber_core_diameter=geometry.fiber_core_diameter,
)
w7x_response = response.model_copy(update={"band_scales": None})
case_scales = ts.fisher_band_scale(
    case_density,
    case_temperatures,
    case_geometry,
    w7x_response,
    ts.absolute_calibration(case_geometry),
    design.thomson_noise_model,
    placement.thomson_signal_scale,
)
case_response = w7x_response.model_copy(update={"band_scales": case_scales})
case_angles = np.degrees(np.asarray(ts.volume_scattering_angles(case_geometry)))
titles = tuple(
    rf"$\theta = {angle:.0f}^\circ$, $T_e = {te / 1e3:g}$ keV"
    for angle, te in zip(case_angles, case_temperatures)
)
figure = plots.plot_bands_on_spectrum(
    case_geometry, case_response, case_temperatures, titles=titles
)
save_pdf(figure, "bands_on_spectrum")
band_rows = []
for case, (angle, temperature, scale) in enumerate(
    zip(case_angles, case_temperatures, np.asarray(case_scales, dtype=float))
):
    low_edges, high_edges = ts.band_edges(case_response, jnp.array([scale]))
    for channel, (low, high) in enumerate(
        zip(low_edges[0].tolist(), high_edges[0].tolist()), start=1
    ):
        band_rows.append(
            {
                "case": case,
                "theta_deg": round(float(angle), 1),
                "te_ev": round(float(temperature), 0),
                "band_scale": round(float(scale), 4),
                "channel": channel,
                "lambda_low_nm": round(low * 1e9, 2),
                "lambda_high_nm": round(high * 1e9, 2),
            }
        )
band_table = pd.DataFrame(band_rows)
band_table.to_csv(OUTPUT_DIR / "bands_on_spectrum_values.csv", index=False)
for case, (angle, te, s) in enumerate(
    zip(case_angles, case_temperatures, np.asarray(case_scales))
):
    print(
        f"case {case}: theta {angle:.1f} deg, Te {te:g} eV, "
        f"ne {case_density[case]:.2e} m^-3, s = {s:.3f}"
    )

## 3. Interferometer chords

The pinned 5 horizontal chords, each given by the two angles of its line of sight on the auxiliary design surface (the variables the placement optimization moves). The middle chord runs along the laser path. Radial information comes from chords at different impact parameters, here different heights.

The first figure draws the chords, each with its impact-parameter normal from the cross-section centre; the second the line-integrated density per chord. The CSV lists each chord's geometry and signal.

In [ ]:
chord_layout_figure, chord_density_figure = plots.plot_interferometer_layout(
    truth.coefficients, chords, electron_density_fn, design
)
save_pdf(chord_layout_figure, "interferometer_chord_layout")
save_pdf(chord_density_figure, "interferometer_line_integrated_density")
toroidal_angle = np.radians(design.toroidal_angle_deg)
plasma, surface = geometry_utils.cross_section_and_surface(
    truth.coefficients, toroidal_angle, design.surface_offset
)
impact_parameters = geometry_utils.chord_impact_parameters(
    chords, surface, (design.laser_surface_angles[0], design.laser_surface_angles[1])
)
surface_angles = np.asarray(design.chord_surface_angles, dtype=float).reshape(-1, 2)
chord_rows = []
for index, chord in enumerate(chords):
    start = np.asarray(chord.chord_start)
    end = np.asarray(chord.chord_end)
    start_rz = np.array([float(np.hypot(start[0], start[1])), float(start[2])])
    end_rz = np.array([float(np.hypot(end[0], end[1])), float(end[2])])
    surface_to_surface = float(np.linalg.norm(end_rz - start_rz))
    span = geometry_utils.segment_curve_span(plasma, start_rz, end_rz)
    density = geometry_utils.chord_line_integrated_density(
        truth.coefficients,
        toroidal_angle,
        plasma,
        start_rz,
        end_rz,
        electron_density_fn,
        n_sample=128,
    )
    sigma = interferometer_utils.compute_interferometer_uncertainty(density)
    chord_rows.append(
        {
            "chord": index + 1,
            "surface_angles_rad": [
                round(float(value), 5) for value in surface_angles[index]
            ],
            "impact_parameter_m": round(float(impact_parameters[index]), 4),
            "surface_to_surface_m": round(surface_to_surface, 4),
            "plasma_path_m": round(
                0.0 if span is None else (span[1] - span[0]) * surface_to_surface, 4
            ),
            "line_integrated_density_m2": f"{density:.4e}",
            "sigma_m2": f"{float(sigma):.4e}",
        }
    )
chord_table = pd.DataFrame(chord_rows)
chord_table.to_csv(OUTPUT_DIR / "interferometer_chord_values.csv", index=False)
print(chord_table.to_string(index=False))
assert (chord_table["plasma_path_m"] > 0.0).all(), "every chord must cross the plasma"
np.testing.assert_allclose(
    chord_table["impact_parameter_m"].abs().min(),
    0.0,
    atol=1e-12,
    err_msg="one chord must sit on the Thomson laser path (d = 0)",
)

## 4. Cross-sections of the thesis

The interferometer chords of section 3 and the Thomson layout of section 1b on the same `wide_high_beta_gp` truth, at the thesis's text width and 9 pt type, sharing their axes height and `(R, Z)` limits so the two panels print side by side at the same scale. The light panel of section 1b is not repeated here.


In [ ]:
# Both panels share the axes height and the R, Z limits, and each is saved at the full
# panel height, cropped only in width, so at natural size in the thesis the two
# cross-sections print identically and every label at its nominal point size.
TEXT_WIDTH_IN = 418.26 / 72.27
PANEL_HEIGHT_IN = 2.85
AXES_HEIGHT_IN = 2.25
AXES_TOP_MARGIN_IN = 0.12
AXES_LEFT_IN = 0.55
COLORBAR_GAP_IN = 0.08
COLORBAR_WIDTH_IN = 0.09
LEGEND_GAP_IN = 0.15
CROP_PAD_IN = 0.03
# The Thomson helper sizes its markers and lines for a full-width figure; at this size
# they are scaled down so the ports and volumes keep their proportion to the
# cross-section.
THOMSON_MARKER_SCALE = 0.5
THOMSON_LINE_SCALE = 0.65
BOUNDARY_STYLE = {"color": "k", "linestyle": "-", "linewidth": 1.0}
SURFACE_STYLE = {"color": "0.45", "linestyle": "--", "linewidth": 0.8}
AXIS_STYLE = {"color": "k", "markersize": 6.0, "markeredgewidth": 1.0}
BOUNDARY_LABEL = r"plasma boundary ($\rho = 1$)"
SURFACE_LABEL = rf"auxiliary design surface ($+{design.surface_offset:.2f}$ m)"

_, design_surface = geometry_utils.cross_section_and_surface(
    truth.coefficients, np.radians(design.toroidal_angle_deg), design.surface_offset
)
surface_curve = np.asarray(design_surface.curve)
margin = 0.05 * float(np.ptp(surface_curve[0]))
x_limits = (float(surface_curve[0].min()) - margin, float(surface_curve[0].max()) + margin)
y_limits = (float(surface_curve[1].min()) - margin, float(surface_curve[1].max()) + margin)
axes_width_in = AXES_HEIGHT_IN * (x_limits[1] - x_limits[0]) / (y_limits[1] - y_limits[0])
axes_bottom_in = PANEL_HEIGHT_IN - AXES_TOP_MARGIN_IN - AXES_HEIGHT_IN


def inches_to_figure(left, bottom, width, height):
    """An axes rectangle given in inches, as fractions of the uncropped canvas."""
    return [
        left / TEXT_WIDTH_IN,
        bottom / PANEL_HEIGHT_IN,
        width / TEXT_WIDTH_IN,
        height / PANEL_HEIGHT_IN,
    ]


def place(axes):
    """Put `axes` at the shared size, position and limits."""
    axes.set_xlim(*x_limits)
    axes.set_ylim(*y_limits)
    axes.set_xlabel(r"$R$ (m)")
    axes.set_ylabel(r"$Z$ (m)")
    axes.grid(False)
    axes.set_position(
        inches_to_figure(AXES_LEFT_IN, axes_bottom_in, axes_width_in, AXES_HEIGHT_IN)
    )


def restyle(axes, styles):
    """Apply `styles[label]` to the one line of `axes` carrying each label."""
    for label, style in styles.items():
        (line,) = [line for line in axes.lines if line.get_label() == label]
        line.set(**style)


def save_cropped(figure, name):
    """Save `figure` cropped to its content in width only; returns that width."""
    figure.canvas.draw()
    content = figure.get_tightbbox(figure.canvas.get_renderer())
    assert content.x0 >= 0.0 and content.x1 <= TEXT_WIDTH_IN, f"{name} overflows"
    crop = matplotlib.transforms.Bbox.from_extents(
        content.x0 - CROP_PAD_IN, 0.0, content.x1 + CROP_PAD_IN, PANEL_HEIGHT_IN
    )
    figure.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches=crop)
    print(f"{name}: {crop.width / TEXT_WIDTH_IN:.3f} of the text width")
    return crop.width


# The house style crops to the ink on save, which would give the two panels different
# heights; each is cropped to its own width at the shared height instead. The context
# keeps the print style from reaching the cells above when the notebook is rerun.
with plt.rc_context():
    pubfig.apply_style(base_font_pt=9.0)
    plt.rcParams["savefig.bbox"] = "standard"

    interferometer_figure, interferometer_axes = plt.subplots(
        figsize=(TEXT_WIDTH_IN, PANEL_HEIGHT_IN)
    )
    plots.draw_interferometer_layout(
        interferometer_axes, truth.coefficients, chords, design
    )
    restyle(
        interferometer_axes,
        {
            BOUNDARY_LABEL: BOUNDARY_STYLE,
            SURFACE_LABEL: SURFACE_STYLE,
            "magnetic axis": AXIS_STYLE,
        },
    )
    place(interferometer_axes)

    # Only the geometry panel is kept; the light panel is section 1b's.
    thomson_figure, thomson_axes = plt.subplots(figsize=(TEXT_WIDTH_IN, PANEL_HEIGHT_IN))
    light_figure, light_axes = plt.subplots()
    plots.plot_chord_layout(
        truth.coefficients,
        geometry,
        signals,
        uncertainties,
        surface_offset=design.surface_offset,
        axes_pair=(thomson_axes, light_axes),
    )
    plt.close(light_figure)
    for line in thomson_axes.lines:
        line.set_linewidth(THOMSON_LINE_SCALE * line.get_linewidth())
        line.set_markersize(THOMSON_MARKER_SCALE * line.get_markersize())
        line.set_markeredgewidth(THOMSON_MARKER_SCALE * line.get_markeredgewidth())
    for collection in thomson_axes.collections:
        collection.set_sizes(THOMSON_MARKER_SCALE**2 * collection.get_sizes())
        collection.set_linewidths(THOMSON_MARKER_SCALE * collection.get_linewidths())
    thomson_surface_label = f"auxiliary design surface (+{design.surface_offset:.3f} m)"
    restyle(
        thomson_axes,
        {
            "plasma boundary (rho = 1)": BOUNDARY_STYLE,
            thomson_surface_label: SURFACE_STYLE,
            "magnetic axis": AXIS_STYLE,
        },
    )
    thomson_axes.get_legend().remove()
    place(thomson_axes)
    (colorbar_axes,) = [axes for axes in thomson_figure.axes if axes is not thomson_axes]
    colorbar_axes.set_position(
        inches_to_figure(
            AXES_LEFT_IN + axes_width_in + COLORBAR_GAP_IN,
            axes_bottom_in,
            COLORBAR_WIDTH_IN,
            AXES_HEIGHT_IN,
        )
    )
    # A colorbar keeps a fixed length-to-width ratio, which would leave it shorter than
    # the axes it labels.
    colorbar_axes.set_box_aspect(AXES_HEIGHT_IN / COLORBAR_WIDTH_IN)

    # The helper's optic labels carry the scattering-angle class and the lens standoff,
    # which the thesis text gives; the legend keeps only what identifies each artist.
    renamed = {
        "plasma boundary (rho = 1)": BOUNDARY_LABEL,
        thomson_surface_label: SURFACE_LABEL,
        "magnetic axis": "magnetic axis",
        "laser path": "laser path",
    }
    handles_by_label = {}
    for handle, label in zip(*thomson_axes.get_legend_handles_labels()):
        if label.startswith(plots.OPTIC_SERIES[0][0]):
            handles_by_label["forward-optic port"] = handle
        elif label.startswith(plots.OPTIC_SERIES[1][0]):
            handles_by_label["backward-optic port"] = handle
        else:
            handles_by_label[renamed[label]] = handle
    legend_order = [
        BOUNDARY_LABEL,
        SURFACE_LABEL,
        "magnetic axis",
        "laser path",
        "forward-optic port",
        "backward-optic port",
    ]
    assert sorted(legend_order) == sorted(handles_by_label), sorted(handles_by_label)

    # The legend starts right of the colorbar's own label, which is measured once drawn.
    thomson_figure.canvas.draw()
    colorbar_right_in = (
        colorbar_axes.get_tightbbox(thomson_figure.canvas.get_renderer())
        .transformed(thomson_figure.dpi_scale_trans.inverted())
        .x1
    )
    thomson_figure.legend(
        [handles_by_label[label] for label in legend_order],
        legend_order,
        loc="center left",
        bbox_to_anchor=(
            (colorbar_right_in + LEGEND_GAP_IN) / TEXT_WIDTH_IN,
            (axes_bottom_in + 0.5 * AXES_HEIGHT_IN) / PANEL_HEIGHT_IN,
        ),
        handlelength=1.2,
        handletextpad=0.5,
        borderaxespad=0.0,
    )

    interferometer_width_in = save_cropped(
        interferometer_figure, "interferometer_cross_section"
    )
    thomson_width_in = save_cropped(thomson_figure, "thomson_cross_section")
assert interferometer_width_in + thomson_width_in < TEXT_WIDTH_IN, "panels overflow"
